In [6]:
import os
import re
import string
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

print("⏳ 1/4 Loading and preprocessing dataset...")
data_path = r'C:\EDP-workspace\Fake-News-Detector\backend\data\cleaned_news_dataset.csv'
df = pd.read_csv(data_path)
df['text'] = df['text'].fillna('')

def clean_text_pipeline(text):
    text = text.lower()
    text = re.sub(r'\(?reuters\)?', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text_pipeline)

print("⏳ 2/4 Vectorizing with TF-IDF (Unigrams + Bigrams)...")
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print("⏳ 3/4 Training optimized Logistic Regression model...")
# Optimized hyperparameters determined via tuning
best_model = LogisticRegression(C=10.0, max_iter=1000, random_state=42)
best_model.fit(X_train_vec, y_train)

# Evaluation
y_pred = best_model.predict(X_test_vec)
y_probs = best_model.predict_proba(X_test_vec)[:, 1]

print("\n" + "="*40)
print(f"✅ Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"✅ ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")
print("="*40)
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['REAL (0)', 'FAKE (1)']))
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n⏳ 4/4 Saving optimized model & vectorizer...")
models_dir = r'C:\EDP-workspace\Fake-News-Detector\backend\models'
os.makedirs(models_dir, exist_ok=True)

joblib.dump(best_model, os.path.join(models_dir, 'fake_news_model.pkl'))
joblib.dump(tfidf, os.path.join(models_dir, 'vectorizer.pkl'))

print("\n🎉 SUCCESS: Model and vectorizer successfully trained, evaluated, and saved to backend/models/!")

⏳ 1/4 Loading and preprocessing dataset...
⏳ 2/4 Vectorizing with TF-IDF (Unigrams + Bigrams)...
⏳ 3/4 Training optimized Logistic Regression model...

✅ Test Accuracy: 98.83%
✅ ROC-AUC Score: 0.9987

--- Classification Report ---
              precision    recall  f1-score   support

    REAL (0)       0.99      0.99      0.99      4242
    FAKE (1)       0.99      0.99      0.99      4696

    accuracy                           0.99      8938
   macro avg       0.99      0.99      0.99      8938
weighted avg       0.99      0.99      0.99      8938

--- Confusion Matrix ---
[[4184   58]
 [  47 4649]]

⏳ 4/4 Saving optimized model & vectorizer...

🎉 SUCCESS: Model and vectorizer successfully trained, evaluated, and saved to backend/models/!
